In [ ]:
%pip install transformers

### Load and Pre-process Dataset

In [ ]:
%pip install datasets

In [ ]:
%pip install transformers

In [ ]:
%pip install tensorflow

In [ ]:
%pip install tf-keras

In [ ]:
%pip install transformers[torch]

In [1]:
from datasets import load_dataset
from transformers import AutoTokenizer

# Load the IMDB dataset
dataset = load_dataset("imdb")

# Load a pre-trained tokenizer (BERT in this case)
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Tokenize the dataset
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

# Apply tokenizer to train and test datasets
tokenized_datasets = dataset.map(tokenize_function, batched=True)

# Set format for PyTorch
tokenized_datasets.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])


Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

### Fine-tune Model

In [2]:
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments

# Load a pre-trained BERT model for sequence classification
model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

# Define training arguments
training_args = TrainingArguments(
    output_dir="./results",          # output directory
    num_train_epochs=3,              # number of training epochs
    per_device_train_batch_size=8,   # batch size for training
    per_device_eval_batch_size=16,   # batch size for evaluation
    evaluation_strategy="epoch",     # evaluation strategy to use
    logging_dir="./logs",            # directory for storing logs
    logging_steps=10,
)

# Define Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
)

# Train the model
trainer.train()


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
c:\Users\mindf\anaconda3\Lib\site-packages\transformers\training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


  0%|          | 0/9375 [00:00<?, ?it/s]

c:\Users\mindf\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'loss': 0.7054, 'grad_norm': 2.176943778991699, 'learning_rate': 4.994666666666667e-05, 'epoch': 0.0}
{'loss': 0.696, 'grad_norm': 3.4448893070220947, 'learning_rate': 4.989333333333334e-05, 'epoch': 0.01}
{'loss': 0.6039, 'grad_norm': 3.172612190246582, 'learning_rate': 4.9840000000000004e-05, 'epoch': 0.01}
{'loss': 0.5418, 'grad_norm': 4.22302770614624, 'learning_rate': 4.978666666666667e-05, 'epoch': 0.01}


KeyboardInterrupt: 

### Save Model

In [3]:
# Save the fine-tuned model locally
model.save_pretrained("./finetuned_model")
tokenizer.save_pretrained("./finetuned_model")


('./finetuned_model\\tokenizer_config.json',
 './finetuned_model\\special_tokens_map.json',
 './finetuned_model\\vocab.txt',
 './finetuned_model\\added_tokens.json',
 './finetuned_model\\tokenizer.json')